In [0]:
from pyspark.sql.functions import col,sum,when,current_timestamp
bronze_df = spark.read.table('fraud_mlops.bronze.paysim_transactions')

print(f'Bronze rows: {bronze_df.count()}')
bronze_df.printSchema()

In [0]:


null_counts = bronze_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in bronze_df.columns])
display(null_counts)

In [0]:
silver_base_df = bronze_df.select(
    col("step"),
    col("type").alias("transaction_type"),
    col("amount"),
    col("nameOrig").alias("origin_account"),
    col("oldbalanceOrg").alias("old_balance_origin"),
    col("newbalanceOrig").alias("new_balance_origin"),
    col("nameDest").alias("destination_account"),
    col("oldbalanceDest").alias("old_balance_destination"),
    col("newbalanceDest").alias("new_balance_destination"),
    col("isFraud").alias("is_fraud"),
    col("isFlaggedFraud").alias("is_flagged_fraud"),
    col("source_file"),
    col("ingested_at")
)

display(silver_base_df.limit(10))
display(silver_base_df.count())

In [0]:
transaction_columns = ["step","transaction_type","amount","origin_account","old_balance_origin","new_balance_origin","destination_account","old_balance_destination","new_balance_destination","is_fraud","is_flagged_fraud"]

silver_base_df = silver_base_df.dropDuplicates(transaction_columns)
display(silver_base_df.count())


In [0]:
silver_base_df.groupBy('transaction_type').count().display()

In [0]:
from pyspark.sql import functions as F

allowed_types = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER"
]

checked_df = (
    silver_base_df

    .withColumn(
        "dq_reason",

        when(
            F.col("step").isNull(),
            "missing_step"
        )

        .when(
            ~col("step").between(1, 744),
            "invalid_step"
        )

        .when(
            col("transaction_type").isNull(),
            "missing_transaction_type"
        )

        .when(
            ~col("transaction_type").isin(allowed_types),
            "invalid_transaction_type"
        )

        .when(
            col("amount").isNull(),
            "missing_amount"
        )

        .when(
            col("amount") < 0,
            "negative_amount"
        )

        .when(
            col("origin_account").isNull(),
            "missing_origin_account"
        )

        .when(
            col("destination_account").isNull(),
            "missing_destination_account"
        )

        .when(
            col("is_fraud").isNotNull()
            & ~col("is_fraud").isin(0, 1),
            "invalid_fraud_label"
        )

        .when(
            col("is_flagged_fraud").isNotNull()
            & ~col("is_flagged_fraud").isin(0, 1),
            "invalid_flagged_fraud"
        )
    )
    .withColumn(
        "label_status",
        when(
            col("is_fraud").isNull(),
            "PENDING"
        ).otherwise("AVAILABLE")
    )
    .withColumn(
        "dq_status",
        when(
            col("dq_reason").isNotNull(),
            "QUARANTINE"
        ).otherwise("VALID")
    )
)

In [0]:
display(checked_df.limit(10))

In [0]:
clean_df = checked_df.filter(col('dq_status') == 'VALID')\
    .drop('dq_reason','dq_status')\
        .withColumn('silver_processed_at',current_timestamp())

quarrentine_df = checked_df.filter(col('dq_status')== 'QUARANTINE')\
    .withColumn('quarantined_at',current_timestamp())

    

 



In [0]:
display(clean_df.limit(10))

In [0]:
display(
    checked_df
    .groupBy("label_status")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
clean_df.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable('fraud_mlops.silver.transactions_clean')

In [0]:
quarrentine_df.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable('fraud_mlops.silver.transactions_quarantine')

In [0]:
silver_df = spark.read.table('fraud_mlops.silver.transactions_clean')
display(silver_df.limit(10))

silver_df.printSchema()

In [0]:
display(silver_df.groupBy("is_fraud","is_flagged_fraud").count())